In [1]:
from __future__ import print_function

import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.activations import relu
from tensorflow.keras.regularizers import l2
from tensorflow.keras.constraints import max_norm
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import mnist, cifar10

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

print("Packages Loaded")

Packages Loaded


In [2]:
# Lets import our MNIST data like we did in lab 3.
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

In [3]:
# Now import Cifar-10 data and process it.
(train, target), (test, test_target) = cifar10.load_data()
# Fill in the rest.

170500096/170498071 [==============================] - 64s 0us/step


In [4]:
# Our baseline model for this lab.
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(784,)))
model.add(Dense(64, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(10, activation='softmax'))
model.compile(loss='categorical_crossentropy',
              optimizer=SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True),
              metrics=['accuracy'])
model.fit(x_train, y_train, epochs=20, batch_size=64, validation_data=(x_test, y_test))

Train on 60000 samples, validate on 10000 samples
Epoch 1/20
60000/60000 [==============================] - 5s 91us/sample - loss: 0.4025 - accuracy: 0.8771 - val_loss: 0.1566 - val_accuracy: 0.9539
Epoch 2/20
60000/60000 [==============================] - 5s 76us/sample - loss: 0.1465 - accuracy: 0.9557 - val_loss: 0.1212 - val_accuracy: 0.9628
Epoch 3/20
60000/60000 [==============================] - 4s 74us/sample - loss: 0.1094 - accuracy: 0.9669 - val_loss: 0.1049 - val_accuracy: 0.9657
Epoch 4/20
60000/60000 [==============================] - 4s 75us/sample - loss: 0.0895 - accuracy: 0.9723 - val_loss: 0.1070 - val_accuracy: 0.9666
Epoch 5/20
60000/60000 [==============================] - 4s 75us/sample - loss: 0.0743 - accuracy: 0.9775 - val_loss: 0.1051 - val_accuracy: 0.9654
Epoch 6/20
60000/60000 [==============================] - 5s 79us/sample - loss: 0.0633 - accuracy: 0.9804 - val_loss: 0.0838 - val_accuracy: 0.9740
Epoch 7/20
60000/60000 [==============================] 

### Now convert this baseline into a Functional Model using keras' Functional Model API.
https://keras.io/getting-started/functional-api-guide/

In [5]:
# Create the functional Baseline here.


### Now we are going to compare our baseline to a shallow ResNet that we talked about in class. 
Pay attention to the changes we have made so far including optimizers, batch size, layer neuron width, and dropout.
Why did we make these changes? 

In [6]:
inputs = tf.keras.Input(shape=(784,), name='img')
x = Dense(128, activation='relu')(inputs)
block_1_output = Dense(128, activation='relu')(x)

x = Dense(128)(block_1_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
block_2_output = tf.keras.layers.add([x, block_1_output])

x = Dense(128)(block_2_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
block_3_output = tf.keras.layers.add([x, block_2_output])

x = Dense(128)(block_3_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
block_4_output = tf.keras.layers.add([x, block_3_output])

x = Dense(128, activation='relu')(block_4_output)
x = Dropout(0.5)(x)
outputs = Dense(10, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='resnet')

model.compile(Adam(amsgrad=True), 'binary_crossentropy', metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=128,
          epochs=20,
          validation_data=(x_test, y_test))

Train on 60000 samples, validate on 10000 samples
Epoch 1/20
60000/60000 [==============================] - 8s 135us/sample - loss: 0.1717 - accuracy: 0.9420 - val_loss: 0.0447 - val_accuracy: 0.9859
Epoch 2/20
60000/60000 [==============================] - 6s 95us/sample - loss: 0.0484 - accuracy: 0.9849 - val_loss: 0.0308 - val_accuracy: 0.9902
Epoch 3/20
60000/60000 [==============================] - 6s 94us/sample - loss: 0.0328 - accuracy: 0.9899 - val_loss: 0.0222 - val_accuracy: 0.9928
Epoch 4/20
60000/60000 [==============================] - 6s 95us/sample - loss: 0.0248 - accuracy: 0.9924 - val_loss: 0.0185 - val_accuracy: 0.9939
Epoch 5/20
60000/60000 [==============================] - 6s 95us/sample - loss: 0.0201 - accuracy: 0.9937 - val_loss: 0.0171 - val_accuracy: 0.9942
Epoch 6/20
60000/60000 [==============================] - 6s 95us/sample - loss: 0.0165 - accuracy: 0.9948 - val_loss: 0.0154 - val_accuracy: 0.9950
Epoch 7/20
60000/60000 [==============================]

### Now lets make a deeper ResNet. Make A network with 10 Residual Blocks. 
How does this affect training speed, accuracy, and stability?

In [7]:
# Create the deep ResNet here.


### Now lets mess with the skip connections. We will make two shallow ResNets that have one and three skip connections.

In [8]:
# Here is a one skip connection block.

x = Dense(128)(block_1_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
block_2_output = tf.keras.layers.add([x, block_1_output])

# Here is a two skip connection block.

x = Dense(128)(block_1_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
block_2_output = tf.keras.layers.add([x, block_1_output])

# Here is a three skip connection block.

x = Dense(128)(block_1_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
block_2_output = tf.keras.layers.add([x, block_1_output])


In [9]:
# Now make a full shallow network with the different sized blocks and compare the models. 


### Now lets create a Functional Model for CIFAR with the same baseline as MNIST.

In [ ]:
# Create the CIFAR baseline here.


### Now lets create a ResNet for CIFAR with the same layers and compare it with MNIST.
Why does our accuracy change so much? 

In [10]:
# Create the CIFAR ResNet here.


### Now lets create a custom ResNet.
Change the layer widths, dropout layers, batch sizes, and skip connections to see what we could do to make the ResNet better. Use the knowledge you gained from Lab 3 to do this.

In [11]:
# Create your own ResNet here.
